In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()
df = pd.read_csv('glassdoor_cleaned_v2.csv')

Saving glassdoor_cleaned_v2.csv to glassdoor_cleaned_v2.csv


In [ ]:
print(df.columns.tolist())
print(df.head())

['review_id', 'rating_date', 'employee_length', 'employee_status', 'employee_type', 'flag_covid', 'flag_featured', 'flags_business_outlook', 'flags_ceo_approval', 'flags_recommend_frend', 'rating_culture_values', 'rating_diversity_inclusion', 'rating_overall', 'rating_work_life', 'summary', 'employee_location', 'employee_job_title', 'advice_to_management', 'review_pros', 'review_cons', 'rating_compensation_benefits', 'rating_senior_leadership', 'rating_career_opportunities']
   review_id               rating_date  employee_length employee_status  \
0          1  2025-06-03T00:00:00.000Z                0         REGULAR   
1          2  2025-05-29T00:00:00.000Z                1        CONTRACT   
2          3  2025-05-28T00:00:00.000Z                1         REGULAR   
3          4  2025-05-26T00:00:00.000Z                2       PART_TIME   
4          5  2025-05-23T00:00:00.000Z                0         REGULAR   

                        employee_type  flag_covid  flag_featured  \
0

In [ ]:
import nltk
import string
from nltk.corpus import stopwords
nltk.download('stopwords')

def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.replace('\u2019', "'")
    tokens = text.lower().translate(str.maketrans('', '', string.punctuation)).split()
    return ' '.join([word for word in tokens if word not in stopwords.words('english')])

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
df['summary_cleaned'] = df['summary'].apply(clean_text)
df['advice_to_management_cleaned'] = df['advice_to_management'].apply(clean_text)
df['review_pros_cleaned'] = df['review_pros'].apply(clean_text)
df['review_cons_cleaned'] = df['review_cons'].apply(clean_text)

print(df[['summary', 'summary_cleaned']].head())
print(df[['review_cons', 'review_cons_cleaned']].head())

           summary  summary_cleaned
0           Amazon           amazon
1             Good             good
2  Good experience  good experience
3             Dhdh             dhdh
4    Good Pay rate    good pay rate
                                         review_cons  \
0      inhumane \ndraining physically &amp; mentally   
1  Sticks rules and completing targets any how, o...   
2  limited opportunities for advancement,No clear...   
3                 Tough on the body\nLabor intensive   
4            Not hiring permanent employees directly   

                                 review_cons_cleaned  
0          inhumane draining physically amp mentally  
1  sticks rules completing targets otherwise get ...  
2  limited opportunities advancementno clear path...  
3                         tough body labor intensive  
4                hiring permanent employees directly  


After applying the initial cleaning function, row 0 of review_cons_cleaned showed “amp” instead of “and” where the original text contained “&”. I realized this came from leftover HTML encoding for an ampersand (&). Since it appears as letters rather than standard punctuation, string.punctuation didn’t catch it, so it needed an extra cleaning step.

I updated the clean_text function to replace “&” with “and” before running the rest of the cleaning steps. This keeps the original meaning instead of leaving a meaningless fragment like “amp” in the cleaned text.

In [ ]:
def clean_text(text):
    if pd.isnull(text):
        return ""
    text = text.replace('\u2019', "'")
    text = text.replace('&amp;', 'and')
    tokens = text.lower().translate(str.maketrans('', '', string.punctuation)).split()
    return ' '.join([word for word in tokens if word not in stopwords.words('english')])

In [ ]:
df['summary_cleaned'] = df['summary'].apply(clean_text)
df['advice_to_management_cleaned'] = df['advice_to_management'].apply(clean_text)
df['review_pros_cleaned'] = df['review_pros'].apply(clean_text)
df['review_cons_cleaned'] = df['review_cons'].apply(clean_text)

print(df[['review_cons', 'review_cons_cleaned']].head())

                                         review_cons  \
0      inhumane \ndraining physically &amp; mentally   
1  Sticks rules and completing targets any how, o...   
2  limited opportunities for advancement,No clear...   
3                 Tough on the body\nLabor intensive   
4            Not hiring permanent employees directly   

                                 review_cons_cleaned  
0              inhumane draining physically mentally  
1  sticks rules completing targets otherwise get ...  
2  limited opportunities advancementno clear path...  
3                         tough body labor intensive  
4                hiring permanent employees directly  


The “&” still appeared unchanged even after I added the fix, so I tested the function directly to figure out where the issue was coming from.


In [ ]:
print(clean_text("test &amp; test"))

test test


In [ ]:
text = "test &amp; test"
print(text.replace('&amp;', 'and'))

test and test


In [ ]:
text = "test &amp; test"
text = text.replace('\u2019', "'")
text = text.replace('&amp;', 'and')
print("After replace:", text)

tokens = text.lower().translate(str.maketrans('', '', string.punctuation)).split()
print("After tokenizing:", tokens)

filtered = [word for word in tokens if word not in stopwords.words('english')]
print("After stopword removal:", filtered)

After replace: test and test
After tokenizing: ['test', 'and', 'test']
After stopword removal: ['test', 'test']


The above revealed that ‘and’ was being correctly created by the replace step, but then removed by stopword filtering, since ‘and’ is a standard English stopword. This means the function was actually working correctly all along, the goal of removing the meaningless ‘amp’ fragment was already being achieved.

This showS that “and” was being correctly created by the replace step but then removed by the stopword filtering because “and” is a common English stopword. So the function was actually working correctly all along. The “amp” fragment was being removed as intended, even though I initially thought the fix wasn’t working.

In [16]:
print(df.columns.tolist())

['review_id', 'rating_date', 'employee_length', 'employee_status', 'employee_type', 'flag_covid', 'flag_featured', 'flags_business_outlook', 'flags_ceo_approval', 'flags_recommend_frend', 'rating_culture_values', 'rating_diversity_inclusion', 'rating_overall', 'rating_work_life', 'summary', 'employee_location', 'employee_job_title', 'advice_to_management', 'review_pros', 'review_cons', 'rating_compensation_benefits', 'rating_senior_leadership', 'rating_career_opportunities', 'summary_cleaned', 'advice_to_management_cleaned', 'review_pros_cleaned', 'review_cons_cleaned']


I checked the cleaned columns against the checklist to make sure the text was lowercase, punctuation was removed, stopwords were filtered, and no unexpected fragments were left behind. I also checked the rows that stood out during the cleaning process to make sure the final output was consistent.

In [17]:
print(df['review_cons_cleaned'].head())

0                inhumane draining physically mentally
1    sticks rules completing targets otherwise get ...
2    limited opportunities advancementno clear path...
3                           tough body labor intensive
4                  hiring permanent employees directly
Name: review_cons_cleaned, dtype: object


In [18]:
print(df['summary_cleaned'].str.contains(r'\bthe\b|\bjust\b|\band\b', regex=True).sum())

0


In [19]:
df.to_csv('glassdoor_cleaned_text.csv', index=False)